# Random Forest - S6E10 Predicting Airline Satisfaction

Second model, alongside `hello.ipynb` (logistic regression baseline). Same data, comparable validation:
a stratified 80/20 split with a fixed seed, and the logistic baseline re-scored on the *same* split below,
so the two models are compared on identical rows.

Why no scaler here: tree ensembles split on thresholds, so monotone rescaling of features changes nothing.
Dropping `StandardScaler` for the forest is deliberate, not an omission.

Experiment log (fill in after each run):

| date | change | val AUC | LB AUC | keep/drop |
|------|--------|---------|--------|-----------|
| 2026-10-16 | logistic baseline (from hello.ipynb), same split | 0.92046 |  |  |
| 2026-10-16 | random forest, 200 trees, defaults | 0.95351 |  |  |


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42


In [2]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')
print(f'Loaded {len(train):,} training rows and {len(test):,} test rows.')
print(train.dtypes)
print(f'Missing values in train:\n{train.isna().sum()[train.isna().sum() > 0]}')


Loaded 699,635 training rows and 299,844 test rows.
id                                     int64
Age                                    int64
Flight Distance                        int64
Inflight wifi service                  int64
Departure/Arrival time convenient      int64
Ease of Online booking                 int64
Gate location                          int64
Food and drink                         int64
Online boarding                        int64
Seat comfort                           int64
Inflight entertainment                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Cleanliness                            int64
Departure Delay in Minutes             int64
Arrival Delay in Minutes             float64
Gender                                   str
Customer Type                            str
Type of Travel                           str
Cla

In [3]:
# Target is stored as True/False; pandas usually infers bool dtype, but convert defensively.
def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
print(f'Target balance:\n{target.value_counts(normalize=True)}')


Target balance:
satisfaction
0    0.556427
1    0.443573
Name: proportion, dtype: float64


## Holdout validation

`hello.ipynb` trains on every labeled row, which leaves no local way to know its real score.
Here we hold out 20% of the rows, stratified on the target, and measure ROC AUC on rows the model never saw.


In [4]:
features = train.drop(columns=['id', 'satisfaction'])
test_features = test.drop(columns=['id'])

categorical_columns = features.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
numeric_columns = features.columns.difference(categorical_columns).tolist()
print(f'{len(numeric_columns)} numeric, {len(categorical_columns)} categorical features: {categorical_columns}')

X_train, X_val, y_train, y_val = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=RANDOM_STATE
)
print(f'Train: {len(X_train):,} rows, validation: {len(X_val):,} rows')


17 numeric, 4 categorical features: ['Gender', 'Customer Type', 'Type of Travel', 'Class']


Train: 559,708 rows, validation: 139,927 rows


## Baseline re-scored on this split

The exact logistic pipeline from `hello.ipynb`, trained and scored on the split above.
This is the number any new model has to beat locally.


In [5]:
baseline_preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_columns),
        ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_columns),
    ]
)
baseline_model = Pipeline([
    ('preprocessor', baseline_preprocessor),
    ('classifier', LogisticRegression(max_iter=200)),
])
baseline_model.fit(X_train, y_train)
baseline_auc = roc_auc_score(y_val, baseline_model.predict_proba(X_val)[:, 1])
print(f'Baseline (logistic) validation ROC AUC: {baseline_auc:.5f}')


Baseline (logistic) validation ROC AUC: 0.92046


## Random forest on the same split

200 trees, otherwise defaults: a bounded budget that fits the machine and tonight's session.
Depth and leaf size are the first levers to test later - one at a time.


In [6]:
rf_preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', SimpleImputer(strategy='median'), numeric_columns),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            # sparse_output=False: RandomForest cannot consume sparse matrices
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), categorical_columns),
    ]
)

rf_model = Pipeline([
    ('preprocessor', rf_preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

rf_model.fit(X_train, y_train)
rf_auc = roc_auc_score(y_val, rf_model.predict_proba(X_val)[:, 1])
print(f'Random forest validation ROC AUC: {rf_auc:.5f}')
print(f'Delta vs logistic baseline: {rf_auc - baseline_auc:+.5f}')


Random forest validation ROC AUC: 0.95351
Delta vs logistic baseline: +0.03305


## Feature importance

Impurity-based importances from the fitted forest. Quick and useful for hypotheses about what to engineer next;
treat the exact ranking with care (it can over-credit high-cardinality numeric features).


In [7]:
feature_names = rf_model.named_steps['preprocessor'].get_feature_names_out()
importances = rf_model.named_steps['classifier'].feature_importances_
importance_table = (
    pd.DataFrame({'feature': feature_names, 'importance': importances})
    .sort_values('importance', ascending=False)
    .head(15)
)
importance_table


,feature,importance
15,numeric__Online boarding,0.161435
12,numeric__Inflight wifi service,0.093323
23,categorical__Class_Business,0.086303
8,numeric__Flight Distance,0.073299
24,categorical__Class_Eco,0.060713
11,numeric__Inflight entertainment,0.059960
0,numeric__Age,0.051514
16,numeric__Seat comfort,0.045336
22,categorical__Type of Travel_Personal Travel,0.043013
21,categorical__Type of Travel_Business travel,0.042588


## Fit on all data and write predictions

If the holdout score beats the baseline, refit the forest on every labeled row (more data usually helps a little)
and write `submission_rf.csv`. Uploading it to Kaggle is your action from your own account - it counts toward the daily limit.


In [8]:
rf_model.fit(features, target)
test_probabilities = rf_model.predict_proba(test_features)[:, 1]

submission = pd.read_csv(data_dir / 'sample_submission.csv')
submission['satisfaction'] = test_probabilities
assert len(submission) == len(test)
assert submission['id'].equals(test['id'])

submission_path = competition_dir / 'submission_rf.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission):,} predictions to {submission_path}.')
submission.head()


Saved 299,844 predictions to submission_rf.csv.


,id,satisfaction
0,699635,0.255
1,699636,0.925
2,699637,0.960
3,699638,0.095
4,699639,0.070
